In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA


In [ ]:
# Remove constant OGs and retain OGs present in at least 5% of species.
data = pd.read_csv('related_files/ml_76_matrix_sens_labeled.tsv', sep='\t', header=0)

for col in data.columns[4:]:
    data[col] = pd.to_numeric(data[col], errors="coerce").fillna(0).astype('int64')

features = data.iloc[:, 4:]
initial_feature_count = features.shape[1]

features = features.loc[:, features.nunique() > 1]

non_zero_fraction = (features != 0).sum() / features.shape[0]
features = features.loc[:, non_zero_fraction >= 0.05]

print(f"Initial OG features: {initial_feature_count}")
print(f"OGs after constant and sparse feature filtering: {features.shape[1]}")


In [ ]:
skews = features.skew()
highly_skewed = skews[skews > 1]

print(f"OGs with skewness > 1: {len(highly_skewed)} / {features.shape[1]}")

if len(highly_skewed) / features.shape[1] > 0.3:
    print("More than 30% of retained OGs have skewness > 1.")
else:
    print("At most 30% of retained OGs have skewness > 1.")


In [ ]:
# Apply log1p to every retained OG, independent of the skewness diagnostic.
features_log = np.log1p(features)
print("log1p matrix shape: ", features.shape)

meta = data.iloc[:, :4]
data_log = pd.concat([meta, features_log], axis=1)

data_log.to_csv('related_files/ml_76_matrix_sens_log1p_cleaned.tsv', sep='\t', index=False)


In [ ]:
print(f"Species: {features.shape[0]}, OG features: {features.shape[1]}")
if features.shape[1] > 5000 and features.shape[0] < 200:
    print("Plotting the first two principal components.")

    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(features)

    pca_df = pd.DataFrame({
        'PC1': pca_result[:, 0],
        'PC2': pca_result[:, 1],
        'label': data['sens_C']
    })

    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='label', palette='Set2')
    plt.title("PCA of OG Features")
    plt.show()
else:
    print("PCA preview skipped for this matrix size.")
